# Descarga de Indice de Precios desde FRED

## Descripcion

Descarga la serie `CPIAUCSL` del Federal Reserve Economic Data (FRED) y la transforma
al schema del modelo de retail del libro: base Ene-2020 = 100, 72 meses (2020-01 a 2025-12).

La salida reemplaza la generacion sintetica de `price_index.yml` con datos reales del BLS.

## Prerequisitos

- Variable Library `API_KEYS` con variable `FRED_API_KEY` configurada (ver `FRED_setup.md`)
- Lakehouse adjunto al notebook

## Columnas de salida

| Columna | Tipo | Descripcion |
|---------|------|-------------|
| IndexKey | int | PK secuencial (1 a 72) |
| MonthStartDate | date | Primer dia del mes |
| IndexValue | decimal | Indice rebaseado (Ene-2020 = 100) |
| InflationRate | decimal | Variacion mensual vs mes anterior (%) |
| SourceLoadTime | date | Fecha de extraccion desde FRED |
| LoadDateTime | date | Fecha de carga al Lakehouse |
| ETLBatchID | string | ID del lote ETL |

In [ ]:
import requests
import pandas as pd
from datetime import date
from pyspark.sql import SparkSession

# ============================================================
# CONFIGURACIÓN
# ============================================================

FRED_SERIES  = "CPIAUCSL"       # CPI mensual, SA, todos consumidores urbanos
DATE_START   = "2020-01-01"
DATE_END     = "2025-12-31"
OUTPUT_PATH  = "Files/Data/Config"
OUTPUT_NAME  = "price_index"
DELTA_SCHEMA = "RetailSales"    # Schema destino de la tabla Delta

# ---- API key desde Variable Library ----
# Variable Library: API_KEYS  |  Variable: FRED_API_KEY
_lib    = notebookutils.variableLibrary.getLibrary("API_KEYS")
api_key = _lib.FRED_API_KEY

if not api_key:
    raise ValueError("FRED_API_KEY vacia en Variable Library API_KEYS. Verifica: Workspace -> API_KEYS -> FRED_API_KEY")
print(f"API key cargada ({len(api_key)} caracteres)")

spark = SparkSession.builder.appName("FREDPriceIndex").getOrCreate()
print("Spark session lista")

In [ ]:
# ============================================================
# DESCARGA DESDE FRED
# ============================================================

url = "https://api.stlouisfed.org/fred/series/observations"
params = {
    "series_id":          FRED_SERIES,
    "api_key":            api_key,
    "file_type":          "json",
    "observation_start":  DATE_START,
    "observation_end":    DATE_END,
    "frequency":          "m",
    "aggregation_method": "avg",
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

observations = response.json().get("observations", [])
if not observations:
    raise ValueError("FRED no devolvio observaciones. Verifica API key y rango de fechas.")

print(f"Observaciones recibidas : {len(observations)}")
print(f"Primera : {observations[0]['date']}  valor = {observations[0]['value']}")
print(f"Ultima  : {observations[-1]['date']}  valor = {observations[-1]['value']}")

In [ ]:
# ============================================================
# TRANSFORMACIÓN
# ============================================================

# --- DataFrame base ---
records = [
    {"date": o["date"], "cpi_raw": float(o["value"])}
    for o in observations
    if o.get("value") != "."          # FRED usa "." para datos faltantes
]

df = pd.DataFrame(records)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# --- Rebase a Ene-2020 = 100 ---
base = df.loc[df["date"] == "2020-01-01", "cpi_raw"].values
if len(base) == 0:
    raise ValueError("No hay dato para Ene-2020. El rebase no puede calcularse.")

df["IndexValue"] = (df["cpi_raw"] / base[0] * 100).round(2)

# --- Tasa de inflación mensual (vs mes anterior) ---
df["InflationRate"] = (
    (df["cpi_raw"] - df["cpi_raw"].shift(1)) / df["cpi_raw"].shift(1) * 100
).round(2)
df.loc[0, "InflationRate"] = 0.0      # Primer mes: sin referencia anterior

# --- Schema final ---
load_date = date.today().isoformat()
batch_id  = f"BATCH-PRICE-INDEX-{date.today().strftime('%Y%m%d')}"

df_final = pd.DataFrame({
    "IndexKey":       range(1, len(df) + 1),
    "MonthStartDate": df["date"].dt.date,
    "IndexValue":     df["IndexValue"],
    "InflationRate":  df["InflationRate"],
    "SourceLoadTime": load_date,
    "LoadDateTime":   load_date,
    "ETLBatchID":     batch_id,
})

print(f"Filas    : {len(df_final)}")
print(f"Rango    : {df_final['MonthStartDate'].min()} a {df_final['MonthStartDate'].max()}")
print(f"Ene-2020 : {df_final.loc[0, 'IndexValue']} (debe ser 100.0)")
print(f"Dic-2025 : {df_final.loc[len(df_final)-1, 'IndexValue']}")
print()
print(df_final[["MonthStartDate", "IndexValue", "InflationRate"]].head(6).to_string(index=False))
# Convertir a PySpark DataFrame
from pyspark.sql import functions as F

for col_name in ["MonthStartDate", "SourceLoadTime", "LoadDateTime"]:
    df_final[col_name] = pd.to_datetime(df_final[col_name])

df_spark = spark.createDataFrame(df_final)

for col_name in ["MonthStartDate", "SourceLoadTime", "LoadDateTime"]:
    df_spark = df_spark.withColumn(col_name, F.col(col_name).cast("date"))

df_spark.printSchema()

In [ ]:
# ============================================================
# VALIDACION
# ============================================================

actual_start = df_final["MonthStartDate"].min()
actual_end   = df_final["MonthStartDate"].max()
expected_rows = (
    (actual_end.year - actual_start.year) * 12
    + (actual_end.month - actual_start.month)
    + 1
)

print(f"Rango disponible : {actual_start} a {actual_end}")
print(f"Filas esperadas  : {expected_rows}  |  obtenidas: {len(df_final)}")

if len(df_final) < expected_rows:
    missing = expected_rows - len(df_final)
    print(f"  Nota: {missing} mes(es) sin dato en FRED (datos federales no disponibles)")

print()

checks = {
    "Al menos 60 meses disponibles":  len(df_final) >= 60,
    "Base Ene-2020 = 100":            abs(df_final.loc[0, "IndexValue"] - 100.0) < 0.01,
    "Sin nulos en IndexValue":        df_final["IndexValue"].notna().all(),
    "Sin nulos en InflationRate":     df_final["InflationRate"].notna().all(),
    "IndexValue crece en el periodo": float(df_final["IndexValue"].iloc[-1]) > float(df_final["IndexValue"].iloc[0]),
}

all_pass = True
for name, passed in checks.items():
    symbol = "OK" if passed else "FAIL"
    print(f"  [{symbol}]  {name}")
    if not passed:
        all_pass = False

if not all_pass:
    raise ValueError("Validacion fallida. Revisa los datos antes de escribir.")

print()
print("Todas las validaciones pasaron.")

In [ ]:
# ============================================================
# ESCRITURA: PARQUET + DELTA TABLE
# ============================================================

final_file  = f"{OUTPUT_PATH}/{OUTPUT_NAME}.parquet"
delta_table = f"{DELTA_SCHEMA}.{OUTPUT_NAME}"

# --- Parquet ---
print(f"Escribiendo Parquet en {final_file} ...")
df_spark.coalesce(1).write.mode("overwrite").option("compression", "snappy").parquet(final_file)
print(f"  OK: {df_spark.count():,} filas")

# --- Delta table ---
print(f"Registrando Delta table {delta_table} ...")
df_spark.write.format("delta").mode("overwrite").saveAsTable(delta_table)
print(f"  OK: {delta_table}")

print()
print(f"Batch ID: {batch_id}")

In [ ]:
# ============================================================
# RESUMEN ANUAL
# ============================================================

df_check = spark.read.parquet(final_file).toPandas()
df_check["year"] = pd.to_datetime(df_check["MonthStartDate"]).dt.year

summary = (
    df_check.groupby("year")
    .agg(
        IndexValue_dic    = ("IndexValue",    "last"),
        InflationRate_avg = ("InflationRate", "mean"),
    )
    .reset_index()
)
summary.columns = ["Anio", "IndexValue Dic", "Inflacion Mensual Prom (%)"]
summary["Cambio Anual (%)"] = (
    (summary["IndexValue Dic"] - summary["IndexValue Dic"].shift(1))
    / summary["IndexValue Dic"].shift(1) * 100
).round(2)
summary["Inflacion Mensual Prom (%)"] = summary["Inflacion Mensual Prom (%)"].round(3)

print("Resumen anual del indice de precios (base Ene-2020 = 100):")
print(summary.to_string(index=False))